# EDA & Zero-Leakage Feature Engineering Pipeline

**Phase 03 — CS114 Final Project: Vietnamese Elderly Scam Detection**  
**Tác giả:** Đặng Chí Thanh (MSSV: 25730067)  
**Ngày:** 2026-09-15  

---

## Mục tiêu

Phân tích khám phá dữ liệu (EDA) toàn diện trên `dataset_preprocessed.csv` (2.193 mẫu, 24 cột) và xây dựng pipeline trích xuất đặc trưng TF-IDF chống rò rỉ dữ liệu (Zero-Leakage).

> **CRITICAL INVARIANT:** Split FIRST → TF-IDF fit on TRAIN ONLY → Scaler fit on TRAIN ONLY → Transform test.

## 0. Setup & Imports

In [ ]:
import sys
import json
import time
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import scipy.sparse as sp

# Resolve project root robustly (works in both script and notebook contexts)
try:
    _here = Path(globals().get("__file__", ".")).resolve()
except:
    _here = Path(".").resolve()
PROJECT_ROOT = _here.parent if _here.name.endswith(".py") else _here
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

from src.feature_engineering import (
    export_feature_matrices, DataSplitter, TextFeatureExtractor,
    AuxiliaryFeatureScaler, assemble_feature_matrix
)
from src.feature_engineering_config import AUX_FEATURE_COLS, EXPECTED_N_FEATURES
from src.utils import RANDOM_STATE

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"Expected feature count: {EXPECTED_N_FEATURES} (5000 TF-IDF + 15 aux)")

## 1. Load & Inspect Dataset

In [ ]:
CSV_PATH = PROJECT_ROOT / 'data' / 'processed' / 'dataset_preprocessed.csv'
df = pd.read_csv(CSV_PATH)
print(f'Dataset shape: {df.shape}')
print(f'Columns ({len(df.columns)}):')
for i, col in enumerate(df.columns, 1):
    print(f'  {i:2d}. {col}')
print(f'Null count: {df.isnull().sum().sum()}')

In [ ]:
print('Label distribution:')
counts = df['label'].value_counts().sort_index()
print(counts)
print(f'Imbalance ratio: {counts[0]/counts[1]:.1f}:1')

## 2. EDA Figure 1: Phân bố Nhãn (Label Distribution)

In [ ]:
from src.eda_plots_part1 import plot_label_distribution
FIGURES_DIR = PROJECT_ROOT / 'reports' / 'figures'
plot_label_distribution(df, FIGURES_DIR)
print('Figure saved to reports/figures/label_distribution.png')

## 3. EDA Figure 2: Phân bố Kịch bản Tâm lý Lừa đảo

In [ ]:
from src.eda_plots_part1 import plot_scam_category_distribution
plot_scam_category_distribution(df, FIGURES_DIR)
print('Scam categories:')
print(df['scam_category'].value_counts())

## 4. EDA Figure 3: So sánh Phân bố Độ dài Văn bản

In [ ]:
from src.eda_plots_part1 import plot_text_length_comparison
plot_text_length_comparison(df, FIGURES_DIR)
for label, name in [(0, 'Normal'), (1, 'Scam')]:
    s = df[df['label'] == label]
    print(f'{name}: text_length mean={s.text_length.mean():.0f}, num_words mean={s.num_words.mean():.0f}')

## 5. EDA Figure 4: Word Clouds (Lừa đảo vs Bình thường)

In [ ]:
from src.eda_plots_part1 import plot_wordclouds
plot_wordclouds(df, FIGURES_DIR)
print('Word clouds saved.')

## 6. EDA Figure 5: Top-20 N-grams Phân tách 2 lớp

In [ ]:
from src.eda_plots_part2 import plot_top_ngrams
plot_top_ngrams(df, FIGURES_DIR, n=20)
print('N-grams chart saved.')

## 7. EDA Figure 6: Correlation Heatmap (15 Aux Features x Label)

In [ ]:
from src.eda_plots_part2 import plot_auxiliary_correlation_heatmap
plot_auxiliary_correlation_heatmap(df, FIGURES_DIR)
corr = df[AUX_FEATURE_COLS + ['label']].corr()['label'].drop('label').sort_values(ascending=False)
print('Top-5 aux features correlated with label:')
print(corr.head())

## 8. EDA Figure 7: Phân bố Nguồn Dữ liệu

In [ ]:
from src.eda_plots_part2 import plot_source_distribution
plot_source_distribution(df, FIGURES_DIR)
print(f'Total unique sources: {df["source"].nunique()}')

## 9. Zero-Leakage Feature Engineering Pipeline

> **Protocol:** (1) Stratified Split 80/20 → (2) TF-IDF fit on X_train_text → (3) StandardScaler fit on X_train_aux → (4) hstack(tfidf, aux) → (5) Export pkl

In [ ]:
print('='*60)
print('STEP 1: Stratified Train/Test Split (80/20, seed=42)')
print('='*60)
splitter = DataSplitter()
X_train_text, X_test_text, X_train_aux, X_test_aux, y_train, y_test = splitter.split(df)
print(f'Train size: {len(y_train)} samples ({y_train.mean()*100:.1f}% scam)')
print(f'Test size:  {len(y_test)} samples ({y_test.mean()*100:.1f}% scam)')
print('Zero-leakage: test indices not seen by vectorizer or scaler')

In [ ]:
print('='*60)
print('STEP 2: TF-IDF Extraction (fit on TRAIN ONLY)')
print('='*60)
text_extractor = TextFeatureExtractor()
X_train_tfidf = text_extractor.fit_transform_train(X_train_text)
X_test_tfidf = text_extractor.transform_test(X_test_text)
print(f'X_train_tfidf: {X_train_tfidf.shape}')
print(f'X_test_tfidf:  {X_test_tfidf.shape}')
print(f'Vocab size: {len(text_extractor.vectorizer.vocabulary_)} unique n-grams')

In [ ]:
print('='*60)
print('STEP 3: Aux Feature Scaling (fit on TRAIN ONLY)')
print('='*60)
aux_scaler = AuxiliaryFeatureScaler()
X_train_aux_scaled = aux_scaler.fit_transform_train(X_train_aux)
X_test_aux_scaled = aux_scaler.transform_test(X_test_aux)
print(f'X_train_aux_scaled: {X_train_aux_scaled.shape}')
print(f'X_test_aux_scaled:  {X_test_aux_scaled.shape}')

In [ ]:
print('='*60)
print('STEP 4: Combine TF-IDF + Aux → Final Matrices')
print('='*60)
X_train = assemble_feature_matrix(X_train_tfidf, X_train_aux_scaled)
X_test = assemble_feature_matrix(X_test_tfidf, X_test_aux_scaled)
print(f'X_train: {X_train.shape}')
print(f'X_test:  {X_test.shape}')
assert X_train.shape[1] == EXPECTED_N_FEATURES
assert X_test.shape[1] == EXPECTED_N_FEATURES
print(f'Feature count verified: {X_train.shape[1]} == {EXPECTED_N_FEATURES}')

## 10. Export Artifacts

In [ ]:
import os
t0 = time.time()
result = export_feature_matrices(df)
elapsed = time.time() - t0
info = result['split_info']
print(f'Pipeline completed in {elapsed:.2f}s')
print(f'n_train={info["n_train"]}, n_test={info["n_test"]}, n_features={info["n_features"]}')
print()
for fname in ['X_train.pkl','X_test.pkl','y_train.pkl','y_test.pkl',
              'tfidf_vectorizer.pkl','aux_scaler.pkl','feature_names.json']:
    p = PROJECT_ROOT / 'data' / 'processed' / fname
    status = 'OK' if p.exists() else 'MISSING'
    size = os.path.getsize(p)/1024 if p.exists() else 0
    print(f'  [{status}] {fname}: {size:.1f} KB')

## 11. Data Integrity Check

In [ ]:
X_train_v = joblib.load(PROJECT_ROOT / 'data' / 'processed' / 'X_train.pkl')
X_test_v = joblib.load(PROJECT_ROOT / 'data' / 'processed' / 'X_test.pkl')
def check_matrix(mat, name):
    d = mat.data
    ok = not np.isnan(d).any() and not np.isinf(d).any()
    print(f'{name}: shape={mat.shape}, clean={ok}')
    return ok
check_matrix(X_train_v, 'X_train')
check_matrix(X_test_v, 'X_test')
print(f'Column match: {X_train_v.shape[1]==X_test_v.shape[1]==EXPECTED_N_FEATURES}')

## 12. Summary

**Phase 03 — COMPLETED** ✅

| Artifact | Shape / Count |
|----------|---------------|
| `X_train.pkl` | (1754, 5015) |
| `X_test.pkl` | (439, 5015) |
| `y_train.pkl` | (1754,) — 19.16% scam |
| `y_test.pkl` | (439,) — 19.13% scam |
| `tfidf_vectorizer.pkl` | 5000 n-grams vocab |
| `aux_scaler.pkl` | 15 aux features |
| `feature_names.json` | 5015 names |
| EDA figures | 7 PNG @ 300 DPI |

**Zero-Leakage Protocol:** Split FIRST → TF-IDF & Scaler fit on TRAIN only → Transform test only.

**Ready for:** Phase 04 (Naive Bayes — Cẩm), Phase 05 (RF — Sơn), Phase 06 (SVM — Cẩm).